# Stage 4, chunks 2 and 3: generate the chat training data

The teacher picked in the trial (`Qwen/Qwen3-14B-AWQ`, notebook 12) writes questions about Urdu and English Wikipedia passages from the training split and answers each question once, with thinking on or off. The mix (`DEFAULT_MIX`, `docs/decisions.md`, 2026-11-26) keeps 10,000 conversations after filtering: Urdu 4,000 grounded with thinking, 2,000 grounded without, 2,000 standalone without; English 1,000, 500 and 500. 15% extra is generated for the filters.

**Run it twice**, with `SHARD = 0` and then `SHARD = 1` (settings cell). Each run is about 3 to 4 hours.

- Accelerator: **GPU T4 x2** (both are used). Internet: **on**. No input, no secret.
- Run with **Save Version > Save & Run All**.
- Send back `tune_data_shard0.zip` and `tune_data_shard1.zip` (Output tab).

In [ ]:
# Install zubaan at one exact commit (only its small core is needed here: the vLLM config generator).
COMMIT = "4c73860"
REPO = f"git+https://github.com/Hashir-Aziz123/zubaan.git@{COMMIT}"
!pip install -q --force-reinstall "zubaan @ {REPO}" tokenizers
!zubaan version

In [ ]:
# CHECK 1: the installed code is the commit above, not a cached older copy.
import json
from importlib import metadata

direct = json.loads(metadata.distribution("zubaan").read_text("direct_url.json"))
installed = direct["vcs_info"]["commit_id"]
print("installed commit:", installed)
assert installed.startswith(COMMIT), f"installed {installed}, wanted {COMMIT}"
print("OK: code version verified")

In [ ]:
import subprocess

# CHECK 2: a GPU that vLLM supports. vLLM needs compute capability 7.0 or higher: a Kaggle T4 is 7.5,
# a P100 is 6.0 and does NOT work. Pick "GPU T4 x2" in the notebook settings.
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
except FileNotFoundError:
    raise SystemExit("No GPU in this session (nvidia-smi is missing). Settings > Accelerator > GPU T4 x2, then run again.") from None
assert r.returncode == 0, "no GPU found: turn on a GPU accelerator (T4) in the notebook settings"
print(r.stdout)
name, cap, mem = [x.strip() for x in r.stdout.splitlines()[0].split(",")]
assert float(cap) >= 7.0, f"{name} has compute capability {cap}; vLLM needs 7.0 or higher. Use a T4."
print("OK: GPU", name, "compute capability", cap, mem)

# Both teachers are split over the two T4s (tensor parallel 2): Qwen3-8B in float16 is about 16 GB
# of weights and does not fit one 16 GB T4.
gpus = [x for x in r.stdout.splitlines() if x.strip()]
assert len(gpus) == 2, f"found {len(gpus)} GPU(s); pick GPU T4 x2 in the notebook settings"
print("OK: two GPUs")

In [ ]:
from pathlib import Path

# Run this notebook TWICE: once with SHARD = 0 and once with SHARD = 1. Each run handles about half
# of the passages (a fixed split by passage id), so the two outputs together are the whole dataset.
SHARD = 0
N_SHARDS = 2

TEACHER = ("Qwen/Qwen3-14B-AWQ", "qwen3-14b-awq")  # picked after the teacher trial (notebook 12)
PER_FORMAT = 2  # questions per passage
OVERGENERATE = 1.15  # 15% spare for the filters
WORKERS = 128  # requests in flight
MAX_MODEL_LEN = 8192
THINK_MAX_TOKENS = 4096  # the trial's 14B needed at most 2,476
NO_THINK_MAX_TOKENS = 2048  # 1,024 cut off 18% of long grounded answers in the trial
SEED = 0
OUT = Path(f"/kaggle/working/data_shard{SHARD}")
OUT.mkdir(parents=True, exist_ok=True)
assert 0 <= SHARD < N_SHARDS

## 1. Evaluation texts, Urdu and English Wikipedia, and this shard's passages

In [ ]:
# Evaluation text first (FLORES devtest and dev), then the Urdu training text, as in Stage 3.
!zubaan data flores --out data
!zubaan data flores --split dev --out data/flores_dev
!zubaan data urdu

In [ ]:
# English Wikipedia paragraphs, built exactly as Stage 3's English replay text (streamed, cleaned,
# guarded against FLORES and Belebele English). About 5 minutes.
from zubaan.adapt.data import build_english_replay, iter_english_wikipedia
from zubaan.adapt.pipeline import belebele_passages
from zubaan.io import read_lines

GUARD = {
    "urd": read_lines("data/flores_urd.txt") + read_lines("data/flores_dev/flores_urd.txt") + belebele_passages("urd_Arab"),
    "eng": read_lines("data/flores_eng.txt") + read_lines("data/flores_dev/flores_eng.txt") + belebele_passages("eng_Latn"),
}
english_meta = build_english_replay(iter_english_wikipedia(), Path("data/english/english_replay.txt"), GUARD["eng"], max_mb=12)
print({k: v for k, v in english_meta.items() if k != "source"})

In [ ]:
import json

from zubaan.tune.generate import DEFAULT_MIX, passage_counts, sample_passages, shard_of, split_by_format, to_jsonl

counts = passage_counts(DEFAULT_MIX, PER_FORMAT, OVERGENERATE)
print("passages needed (all shards):", counts)
corpora = {"urd": "data/urdu/urdu_train.txt", "eng": "data/english/english_replay.txt"}
plan, passage_stats = {}, {}
for lang, path in corpora.items():
    need = {fmt: counts.get((lang, fmt), 0) for fmt in ("grounded", "standalone")}
    # Every shard samples the same full list (same seed), then keeps its own part.
    passages, passage_stats[lang] = sample_passages(read_lines(path), sum(need.values()), GUARD[lang], lang=lang, seed=SEED)
    for fmt, ps in split_by_format(passages, need).items():
        plan[(lang, fmt)] = [p for p in ps if shard_of(p.id, N_SHARDS) == SHARD]
mine = [p for ps in plan.values() for p in ps]
(OUT / "passages.jsonl").write_text(to_jsonl(mine), encoding="utf-8")
(OUT / "passage_stats.json").write_text(json.dumps(passage_stats, indent=1))
print({f"{k[0]} {k[1]}": len(v) for k, v in plan.items()}, "passages in this shard")

## 2. Install vLLM (isolated environment)

In [ ]:
# vLLM goes into its own Python 3.12 environment so it cannot replace Kaggle's torch. Takes 5 to 10 minutes.
!pip install -q uv
!uv venv --python 3.12 /tmp/vllm-env
!uv pip install --python /tmp/vllm-env/bin/python vllm
VLLM_VERSION = subprocess.run(
    ["/tmp/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"], capture_output=True, text=True
).stdout.strip()
print("vLLM", VLLM_VERSION)
assert VLLM_VERSION, "vLLM did not install"

## 3. Questions, then answers (saved as they arrive)

In [ ]:
import os
import signal
import time
import urllib.request

from zubaan.tune.generate import teacher_serve_command

ENV = dict(os.environ, PATH="/tmp/vllm-env/bin:" + os.environ["PATH"])


def gpu_used_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def start_teacher(model, served):
    log = OUT / f"vllm_{served}.log"
    cmd = teacher_serve_command(model, served, tensor_parallel=2, max_model_len=MAX_MODEL_LEN)
    print(" ".join(cmd))
    with open(log, "w") as log_file:  # the server keeps its own copy of the handle
        proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
    for _ in range(240):  # up to 40 minutes: download, load, CUDA graph capture
        time.sleep(10)
        if proc.poll() is not None:
            break
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=10) as r:
                if served in r.read().decode():
                    print("server up:", served)
                    return proc
        except Exception:
            pass
    print(log.read_text()[-4000:])
    raise RuntimeError(f"vLLM did not start for {model}: see {log}")


def stop_teacher(proc):
    os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    for _ in range(60):
        time.sleep(5)
        if proc.poll() is not None and max(gpu_used_mb()) < 1000:
            break
    print("stopped; GPU memory in use (MB):", gpu_used_mb())

In [ ]:
from zubaan.tune.client import ChatClient
from zubaan.tune.generate import write_questions

model, served = TEACHER
proc = start_teacher(model, served)
client = ChatClient("http://127.0.0.1:8000", served)
t0 = time.perf_counter()
questions, question_stats = [], {}
for (lang, fmt), ps in plan.items():
    qs, st = write_questions(client, ps, per_format=PER_FORMAT, formats=[fmt], workers=WORKERS)
    questions += qs
    question_stats[f"{lang} {fmt}"] = {k: v for k, v in st.items() if k != "failures"}
question_stats["seconds"] = round(time.perf_counter() - t0, 1)
(OUT / "questions.jsonl").write_text(to_jsonl(questions), encoding="utf-8")
(OUT / "question_stats.json").write_text(json.dumps(question_stats, indent=1))
print(question_stats)
for q in questions[:3]:
    print(q.lang, q.format, q.kind, "|", q.text)

In [ ]:
import dataclasses
import threading

from zubaan.tune.client import QWEN3_NO_THINKING, QWEN3_THINKING
from zubaan.tune.generate import assign_modes, generate_answers

jobs = assign_modes(questions, DEFAULT_MIX, seed=SEED)
print(len(jobs), "answers to write;", sum(t for _, t in jobs), "with thinking")
lock = threading.Lock()
progress = {"n": 0, "tokens": 0, "t0": time.perf_counter()}
answers_file = (OUT / "answers.jsonl").open("w", encoding="utf-8")


def save(rec):
    # Written as each answer arrives, so the file holds everything finished if the session stops.
    with lock:
        answers_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
        answers_file.flush()
        progress["n"] += 1
        progress["tokens"] += rec["completion_tokens"] or 0
        if progress["n"] % 200 == 0:
            secs = time.perf_counter() - progress["t0"]
            print(f"{progress['n']}/{len(jobs)} answers, {progress['tokens'] / secs:.0f} tokens/s, {secs / 60:.0f} min")


try:
    records = generate_answers(
        client, questions, mine, teacher=served, jobs=jobs, workers=WORKERS, on_record=save,
        think_sampling=dataclasses.replace(QWEN3_THINKING, max_tokens=THINK_MAX_TOKENS),
        no_think_sampling=dataclasses.replace(QWEN3_NO_THINKING, max_tokens=NO_THINK_MAX_TOKENS),
    )
finally:
    answers_file.close()
    stop_teacher(proc)
answer_seconds = round(time.perf_counter() - progress["t0"], 1)

## 4. Counts and the zip

In [ ]:
import shutil

from zubaan.tune.generate import summarize_answers

summary = {
    "shard": SHARD,
    "n_shards": N_SHARDS,
    "teacher": TEACHER[0],
    "vllm": VLLM_VERSION,
    "gpu": name,
    "settings": {"per_format": PER_FORMAT, "overgenerate": OVERGENERATE, "think_max_tokens": THINK_MAX_TOKENS,
                 "no_think_max_tokens": NO_THINK_MAX_TOKENS, "workers": WORKERS, "seed": SEED},
    "english_replay": {k: v for k, v in english_meta.items() if k != "source"},
    "questions": question_stats,
    "answers": len(records),
    "answer_seconds": answer_seconds,
    "new_tokens": progress["tokens"],
    "groups": summarize_answers(records),
}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
for key, g in summary["groups"].items():
    print(key.ljust(40), {k: g[k] for k in ("n", "failed_requests", "answered", "hit_limit", "loops", "urdu_answers", "source_in_answer")})
shutil.make_archive(f"/kaggle/working/tune_data_shard{SHARD}", "zip", OUT)
print(f"download /kaggle/working/tune_data_shard{SHARD}.zip from the Output tab")